In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model

MODEL_PATH = r"C:\Users\Ayush\Alzheimer_Project\models\baseline_cnn_ready_for_gradcam.h5"

# Load final model
model = load_model(MODEL_PATH, compile=False)

# Freeze ALL layers
for layer in model.layers:
    layer.trainable = False

# Recompile (for safety, no training will happen)
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Save frozen model (overwrite same file)
model.save(MODEL_PATH)

print("✅ FINAL MODEL FROZEN AND LOCKED")
model.summary()


✅ FINAL MODEL FROZEN AND LOCKED


Model: "functional_40"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 4)              │    23,907,908 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,907,908 (91.20 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,907,908 (91.20 MB)

In [3]:
import os

MODEL_DIR = r"C:\Users\Ayush\Alzheimer_Project\models"

print("📁 Files in models directory:\n")
for f in os.listdir(MODEL_DIR):
    print(f)


📁 Files in models directory:

baseline_cnn_best.h5
baseline_cnn_final.h5
baseline_cnn_fixed.h5
baseline_cnn_ready_for_gradcam.h5
baseline_cnn_unwrapped.h5
mobilenetv2_best.h5
mobilenetv2_finetuned.h5
mobilenetv2_head.h5


In [6]:
# ================================
# PHASE 3 – STEP 2: 5-FOLD CROSS-VALIDATION
# (Frozen MobileNetV2 Model)
# ================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --------------------------------------------------
# Paths
# --------------------------------------------------
MODEL_PATH = r"C:\Users\Ayush\Alzheimer_Project\models\baseline_cnn_ready_for_gradcam.h5"
DATASET_DIR = r"C:\Users\Ayush\Alzheimer_Project\Alzheimer_Dataset"
OUTPUT_DIR = r"C:\Users\Ayush\Alzheimer_Project\outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------------------------------
# Load frozen model
# --------------------------------------------------
model = tf.keras.models.load_model(MODEL_PATH)
model.trainable = False  # safety lock

print("✅ Frozen model loaded for cross-validation")

# --------------------------------------------------
# Load image paths and labels
# --------------------------------------------------
class_names = sorted(os.listdir(DATASET_DIR))
class_to_idx = {cls: i for i, cls in enumerate(class_names)}

image_paths = []
labels = []

for cls in class_names:
    cls_path = os.path.join(DATASET_DIR, cls)
    for img in os.listdir(cls_path):
        image_paths.append(os.path.join(cls_path, img))
        labels.append(class_to_idx[cls])

image_paths = np.array(image_paths)
labels = np.array(labels)

print(f"Total samples: {len(image_paths)}")
print("Classes:", class_to_idx)

# --------------------------------------------------
# Image generator (NO augmentation for CV)
# --------------------------------------------------
datagen = ImageDataGenerator(rescale=1./255)

def create_generator(paths, labels, batch_size=32, shuffle=False):
    df = pd.DataFrame({"filename": paths, "class": labels.astype(str)})
    return datagen.flow_from_dataframe(
        df,
        x_col="filename",
        y_col="class",
        target_size=(224, 224),
        class_mode="categorical",
        batch_size=batch_size,
        shuffle=shuffle
    )

# --------------------------------------------------
# 5-Fold Stratified Cross-Validation
# --------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels), start=1):
    print(f"\n🔁 Fold {fold}/5")

    val_gen = create_generator(
        image_paths[val_idx],
        labels[val_idx],
        shuffle=False
    )

    preds = model.predict(val_gen, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = labels[val_idx]

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    fold_results.append([fold, acc, prec, rec, f1])

    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}")

# --------------------------------------------------
# Save results
# --------------------------------------------------
cv_df = pd.DataFrame(
    fold_results,
    columns=["Fold", "Accuracy", "Precision", "Recall", "F1-Score"]
)

cv_df.loc["Mean"] = ["—"] + cv_df.iloc[:, 1:].mean().tolist()
cv_df.loc["Std"] = ["—"] + cv_df.iloc[:, 1:].std().tolist()

csv_path = os.path.join(OUTPUT_DIR, "phase3_5fold_crossval_results.csv")
cv_df.to_csv(csv_path, index=False)

print("\n✅ 5-Fold Cross-Validation Completed")
print(cv_df)
print(f"\n📁 Results saved to: {csv_path}")


✅ Frozen model loaded for cross-validation
Total samples: 86437
Classes: {'Mild Dementia': 0, 'Moderate Dementia': 1, 'Non Demented': 2, 'Very mild Dementia': 3}

🔁 Fold 1/5
Found 17288 validated image filenames belonging to 4 classes.
Accuracy: 0.9143, F1: 0.9147

🔁 Fold 2/5
Found 17288 validated image filenames belonging to 4 classes.
Accuracy: 0.9157, F1: 0.9165

🔁 Fold 3/5
Found 17287 validated image filenames belonging to 4 classes.
Accuracy: 0.9172, F1: 0.9184

🔁 Fold 4/5
Found 17287 validated image filenames belonging to 4 classes.
Accuracy: 0.9169, F1: 0.9175

🔁 Fold 5/5
Found 17287 validated image filenames belonging to 4 classes.
Accuracy: 0.9132, F1: 0.9141

✅ 5-Fold Cross-Validation Completed
     Fold  Accuracy  Precision    Recall  F1-Score
0       1  0.914334   0.919227  0.914334  0.914719
1       2  0.915722   0.920800  0.915722  0.916498
2       3  0.917221   0.923085  0.917221  0.918384
3       4  0.916874   0.921920  0.916874  0.917549
4       5  0.913172   0.918496 

In [1]:
import tensorflow as tf
import os

# Paths
MODEL_DIR = r"C:\Users\Ayush\Alzheimer_Project\models"
TFLITE_DIR = r"C:\Users\Ayush\Alzheimer_Project\outputs\tflite"
os.makedirs(TFLITE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "mobilenetv2_finetuned.h5")
TFLITE_PATH = os.path.join(TFLITE_DIR, "mobilenetv2_ad_classifier.tflite")

# Load model
model = tf.keras.models.load_model(MODEL_PATH)
model.trainable = False

print("✅ Model loaded for TFLite conversion")

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimization (important for deployment)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

# Save TFLite model
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

print("✅ TFLite model saved at:", TFLITE_PATH)
print("📦 TFLite model size (MB):", round(len(tflite_model) / (1024*1024), 2))


✅ Model loaded for TFLite conversion
INFO:tensorflow:Assets written to: C:\Users\Ayush\AppData\Local\Temp\tmp8ygc_3jg\assets


INFO:tensorflow:Assets written to: C:\Users\Ayush\AppData\Local\Temp\tmp8ygc_3jg\assets


Saved artifact at 'C:\Users\Ayush\AppData\Local\Temp\tmp8ygc_3jg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1859352719824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1859352720400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1859352722128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1859352721744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1859352720592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1859352722320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1859352722896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1859352723088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1859352720976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1859352721552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1859

In [2]:
import numpy as np
import cv2
import tensorflow as tf

# Load TFLite model
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input shape:", input_details[0]['shape'])
print("Output shape:", output_details[0]['shape'])

# Dummy test image
dummy_img = np.random.rand(224, 224, 3).astype(np.float32)
dummy_img = np.expand_dims(dummy_img, axis=0)

interpreter.set_tensor(input_details[0]['index'], dummy_img)
interpreter.invoke()

output = interpreter.get_tensor(output_details[0]['index'])
print("Dummy prediction:", output)


C:\Users\Ayush\anaconda3\envs\alzheimer\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Input shape: [  1 224 224   3]
Output shape: [1 4]
Dummy prediction: [[1.7833197e-04 3.6540321e-08 9.9740160e-01 2.4200329e-03]]


In [3]:
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,263,110 (8.63 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,263,108 (8.63 MB)

 Optimizer params: 2 (12.00 B)

In [4]:
pip install plotly


   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.9 MB 29.1 MB/s eta 0:00:01
   ------------------ --------------------- 4.5/9.9 MB 14.9 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.9 MB 11.4 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/9.9 MB 8.8 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/9.9 MB 8.8 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/9.9 MB 5.6 MB/s eta 0:00:01
   ------------------------------ --------- 7.6/9.9 MB 5.5 MB/s eta 0:00:01
   ---------------------------------- ----- 8.7/9.9 MB 5.3 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.9 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 4.8 MB/s  0:00:02
Note: you may need to restart the kernel to use updated packages.


In [1]:
import time
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.platypus import Image as RLImage
from reportlab.platypus import Table
from reportlab.platypus import TableStyle
from reportlab.platypus import ListFlowable, ListItem
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics
import tempfile
import os

In [2]:
pip install reportlab

Note: you may need to restart the kernel to use updated packages.
